# DeFi利回り 探索的分析

`src/yields.py`(APYスナップショット)と`src/scoring.py`(安定性スコアリング)の出力を使って、
チェーン別のAPY分布や、スコアリングでどのプールが減点・除外されているかを可視化する。

読み取り専用。ウォレット接続や送金は行わない。

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

sys.path.append(str(Path("..") / "src"))
from yields import fetch_pools, top_pools_by_apy, DATA_DIR

pools = fetch_pools()
len(pools)

## チェーン別のステーブルコインAPY分布

TVLが薄いプールは除外し(既定: 100万ドル以上)、チェーンごとの中央値APYを比較する。

In [ ]:
stable = pools[(pools["tvlUsd"] >= 1_000_000) & (pools["stablecoin"])]
chain_median_apy = (
    stable.groupby("chain")["apy"]
    .median()
    .sort_values(ascending=False)
    .head(15)
)

fig, ax = plt.subplots(figsize=(8, 5))
chain_median_apy.plot(kind="barh", ax=ax)
ax.invert_yaxis()
ax.set_xlabel("median APY (%)")
ax.set_title("チェーン別 ステーブルコインプールの中央値APY(上位15チェーン)")
fig.tight_layout()

## 現時点のAPY上位プール(生データ、安定性未考慮)

In [ ]:
top_pools_by_apy(pools, stablecoin_only=True, top_n=20)

## 安定性を加味したスコアリング結果

`src/scoring.py`を実行済みであれば`data/ranked_pools.csv`が存在するはずなので、それを読み込む
(毎回全プールの履歴をAPIから取り直すと時間がかかるため)。無ければ先に
`python src/scoring.py`を実行すること。

In [ ]:
ranked_path = Path(DATA_DIR) / "ranked_pools.csv"
if not ranked_path.exists():
    raise FileNotFoundError(
        f"{ranked_path} が見つからない。先に `python src/scoring.py` を実行してください。"
    )
ranked = pd.read_csv(ranked_path)
ranked["flag"].value_counts()

In [ ]:
top15 = ranked.dropna(subset=["score"]).sort_values("score", ascending=False).head(15)
top15

## 「見せかけの高APY」を可視化

横軸に現在のAPY(current_apy)、縦軸にスコアを取ると、現在APYは高いのにスコアが伸びない
(=ボラティリティやTVL急減で減点された)プールが視覚的に分かる。

In [ ]:
FLAG_COLORS = {
    "ok": "tab:blue",
    "tvl_drawdown_risk": "tab:orange",
    "past_hack_risk": "tab:red",
    "high_volatility_excluded": "tab:gray",
    "hacked_protocol_excluded": "tab:gray",
    "category_excluded": "tab:gray",
}

scored = ranked.dropna(subset=["score"])
# 未知のflagが将来増えてもKeyErrorではなくグレーにフォールバックする(NaNのまま
# matplotlibに渡すとValueErrorになるため)。
colors = scored["flag"].map(FLAG_COLORS).fillna("tab:gray")

fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(scored["current_apy"], scored["score"], c=colors, alpha=0.7)
lims = [0, max(scored["current_apy"].max(), scored["score"].max()) * 1.05]
ax.plot(lims, lims, linestyle="--", color="gray", linewidth=1, label="score == current_apy")
ax.set_xlabel("current APY (%)")
ax.set_ylabel("安定性加味スコア")
ax.set_title("現在APY vs 安定性スコア(オレンジ = TVL急減リスク、赤 = 過去ハック歴で減点)")
ax.legend()
fig.tight_layout()